# KNN Classification Analysis for All Datasets

This notebook applies KNN classification with feature selection and cross-validation to all three datasets:
1. **Telco Customer Churn** - Predicting customer churn
2. **Online Shoppers Intention** - Predicting purchase intention
3. **Bank Marketing** - Predicting term deposit subscription

## Import Required Libraries

In [ ]:
# import relevant libraries
import numpy as np               # math, handling of vectors, matrices, tensors
import pandas as pd              # data frames, data preprocessing
import matplotlib.pyplot as plt  # visualization
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import SelectKBest, SelectPercentile, chi2
from sklearn.neighbors import KNeighborsClassifier   # for KNN classifier
from sklearn.model_selection import train_test_split, cross_val_score # for random split in training and validation
from sklearn.pipeline import Pipeline
from sklearn import preprocessing                    # for feature normalization
from sklearn.preprocessing import MinMaxScaler       # for scaling continuous features for Chi2
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("Libraries imported successfully!")

---
## 1. Telco Customer Churn Dataset

### Load and Explore Data

In [ ]:
# Load Telco Customer Churn dataset
telco_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df_telco = pd.read_csv(telco_path)

print(f"Dataset shape: {df_telco.shape}")
print(f"\nFirst few rows:")
print(df_telco.head())
print(f"\nColumn names:")
print(df_telco.columns.tolist())
print(f"\nData types:")
print(df_telco.dtypes)
print(f"\nMissing values:")
print(df_telco.isnull().sum())

### Preprocessing and Feature Engineering

In [ ]:
# Separate features and response
X_telco = df_telco.iloc[:, 2:20]  # Feature columns
y_telco = df_telco['Churn']        # Target variable

print(f"Features shape: {X_telco.shape}")
print(f"Target shape: {y_telco.shape}")

# Convert TotalCharges to numeric (it's stored as string)
X_telco['TotalCharges'] = pd.to_numeric(X_telco['TotalCharges'], errors='coerce')
X_telco['TotalCharges'] = X_telco['TotalCharges'].fillna(X_telco['TotalCharges'].median())

# Re-identify continuous and categorical features after conversion
telco_continuous = X_telco.select_dtypes(include=[np.number]).columns.tolist()
telco_categorical = X_telco.select_dtypes(include=['object']).columns.tolist()

print(f"\nContinuous features ({len(telco_continuous)}): {telco_continuous}")
print(f"Categorical features ({len(telco_categorical)}): {telco_categorical}")

# Scale continuous features with MinMaxScaler
if telco_continuous:
    print(f"\nScaling {len(telco_continuous)} continuous features...")
    X_telco[telco_continuous] = MinMaxScaler().fit_transform(X_telco[telco_continuous])

# Encode ONLY categorical variables
if telco_categorical:
    print(f"Encoding {len(telco_categorical)} categorical features...")
    X_telco = pd.get_dummies(X_telco, columns=telco_categorical, drop_first=True, dtype=float)
else:
    print("No categorical features - converting all to float")
    X_telco = X_telco.astype(float)

print(f"Shape after encoding: {X_telco.shape}")

# Ensure all values are numeric and non-negative
X_telco = X_telco.fillna(0)
X_telco = X_telco.clip(lower=0)

# Encode target variable
y_telco = (y_telco == 'Yes').astype(int)  # 'Yes' -> 1, 'No' -> 0

print(f"\nFinal Features after encoding shape: {X_telco.shape}")
print(f"Feature names (first 20): {X_telco.columns.tolist()[:20]}")
print(f"Target variable distribution:")
print(y_telco.value_counts())

### Feature Selection

In [ ]:
# Feature selection with SelectPercentile
X_telco_selected = SelectPercentile(score_func=chi2, percentile=10).fit_transform(X_telco, y_telco)
print(f"Features after selection shape: {X_telco_selected.shape}")

# Split train and validation data
X_telco_train, X_telco_valid, y_telco_train, y_telco_valid = train_test_split(
    X_telco_selected, y_telco, train_size=0.7, stratify=y_telco
)

print(f"\nTraining set size: {X_telco_train.shape[0]}")
print(f"Validation set size: {X_telco_valid.shape[0]}")

### Scaling and Model Training

In [ ]:
# Scale to mean = 0 and standard deviation = 1
telco_scaler = preprocessing.StandardScaler().fit(X_telco_train)
X_telco_train_scaled = telco_scaler.transform(X_telco_train)
X_telco_valid_scaled = telco_scaler.transform(X_telco_valid)

# Create and train KNN model
knn_telco = KNeighborsClassifier(n_neighbors=7)
knn_telco.fit(X_telco_train_scaled, y_telco_train)

# Cross-validation
cv_scores_telco = cross_val_score(knn_telco, X_telco_train_scaled, y_telco_train, cv=5)
print(f"Cross-validation scores: {cv_scores_telco}")
print(f"Mean CV score: {cv_scores_telco.mean():.4f} (+/- {cv_scores_telco.std():.4f})")

# Calculate accuracy
acc_telco_train = knn_telco.score(X_telco_train_scaled, y_telco_train) * 100
acc_telco_valid = knn_telco.score(X_telco_valid_scaled, y_telco_valid) * 100
print(f"\nAccuracy on training data: {acc_telco_train:.2f}%")
print(f"Accuracy on validation data: {acc_telco_valid:.2f}%")

### Model Evaluation

In [ ]:
# Predictions
y_telco_pred = knn_telco.predict(X_telco_valid_scaled)

# Confusion Matrix
print("Confusion Matrix:")
cm_telco = pd.crosstab(y_telco_valid, y_telco_pred, rownames=['Actual'], colnames=['Predicted'])
print(cm_telco)

# Visualization
cm_telco_values = confusion_matrix(y_telco_valid, y_telco_pred, labels=knn_telco.classes_)
plt.figure(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm_telco_values, display_labels=['No Churn', 'Churn']).plot(cmap='BuPu')
plt.title('Telco Customer Churn - Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

---
## 2. Online Shoppers Intention Dataset

### Load and Explore Data

In [ ]:
# Load Online Shoppers dataset
online_path = "online_shoppers_intention.csv"
df_online = pd.read_csv(online_path)

print(f"Dataset shape: {df_online.shape}")
print(f"\nFirst few rows:")
print(df_online.head())
print(f"\nColumn names:")
print(df_online.columns.tolist())
print(f"\nData types:")
print(df_online.dtypes)
print(f"\nMissing values:")
print(df_online.isnull().sum())

### Preprocessing and Feature Engineering

In [ ]:
# Identify target variable (typically 'Revenue' or 'Purchase')
# Assuming the last column is the target
target_col_online = df_online.columns[-1]
print(f"Target column: {target_col_online}")

# Separate features and response
X_online = df_online.drop(columns=[target_col_online])
y_online = df_online[target_col_online]

print(f"\nFeatures shape: {X_online.shape}")
print(f"Target shape: {y_online.shape}")
print(f"Target variable distribution:")
print(y_online.value_counts())

# Get continuous and categorical columns
online_continuous = X_online.select_dtypes(include=[np.number]).columns.tolist()
online_categorical = X_online.select_dtypes(include=['object']).columns.tolist()

print(f"\nContinuous features: {online_continuous}")
print(f"Categorical features: {online_categorical}")

In [ ]:
# Scale continuous features with MinMaxScaler
if online_continuous:
    X_online[online_continuous] = MinMaxScaler().fit_transform(X_online[online_continuous])

# Encode categorical variables to numeric (one-hot encoding)
if online_categorical:
    X_online = pd.get_dummies(X_online, drop_first=True)

# Convert ALL columns to numeric explicitly
for col in X_online.columns:
    X_online[col] = pd.to_numeric(X_online[col], errors='coerce')

# Fill NaN values and clip negatives
X_online = X_online.fillna(0)
X_online = X_online.clip(lower=0)

# Encode target variable if categorical
if y_online.dtype == 'object':
    y_online = (y_online == y_online.unique()[1]).astype(int)

print(f"Features after encoding shape: {X_online.shape}")
print(f"Target variable distribution:")
print(y_online.value_counts())

### Feature Selection

In [ ]:
# Feature selection with SelectPercentile
X_online_selected = SelectPercentile(score_func=chi2, percentile=10).fit_transform(X_online, y_online)
print(f"Features after selection shape: {X_online_selected.shape}")

# Split train and validation data
X_online_train, X_online_valid, y_online_train, y_online_valid = train_test_split(
    X_online_selected, y_online, train_size=0.7, stratify=y_online
)

print(f"\nTraining set size: {X_online_train.shape[0]}")
print(f"Validation set size: {X_online_valid.shape[0]}")

### Scaling and Model Training

In [ ]:
# Scale to mean = 0 and standard deviation = 1
online_scaler = preprocessing.StandardScaler().fit(X_online_train)
X_online_train_scaled = online_scaler.transform(X_online_train)
X_online_valid_scaled = online_scaler.transform(X_online_valid)

# Create and train KNN model
knn_online = KNeighborsClassifier(n_neighbors=7)
knn_online.fit(X_online_train_scaled, y_online_train)

# Cross-validation
cv_scores_online = cross_val_score(knn_online, X_online_train_scaled, y_online_train, cv=5)
print(f"Cross-validation scores: {cv_scores_online}")
print(f"Mean CV score: {cv_scores_online.mean():.4f} (+/- {cv_scores_online.std():.4f})")

# Calculate accuracy
acc_online_train = knn_online.score(X_online_train_scaled, y_online_train) * 100
acc_online_valid = knn_online.score(X_online_valid_scaled, y_online_valid) * 100
print(f"\nAccuracy on training data: {acc_online_train:.2f}%")
print(f"Accuracy on validation data: {acc_online_valid:.2f}%")

### Model Evaluation

In [ ]:
# Predictions
y_online_pred = knn_online.predict(X_online_valid_scaled)

# Confusion Matrix
print("Confusion Matrix:")
cm_online = pd.crosstab(y_online_valid, y_online_pred, rownames=['Actual'], colnames=['Predicted'])
print(cm_online)

# Visualization
cm_online_values = confusion_matrix(y_online_valid, y_online_pred, labels=knn_online.classes_)
plt.figure(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm_online_values, display_labels=['No Purchase', 'Purchase']).plot(cmap='BuPu')
plt.title('Online Shoppers Intention - Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

---
## 3. Bank Marketing Dataset

### Load and Explore Data

In [ ]:
# Load Bank Marketing dataset
bank_path = "bank-additional-full.csv"
# Try with semicolon delimiter (common for European datasets)
df_bank = pd.read_csv(bank_path, sep=';')

print(f"Dataset shape: {df_bank.shape}")
print(f"\nFirst few rows:")
print(df_bank.head())
print(f"\nColumn names:")
print(df_bank.columns.tolist())
print(f"\nData types:")
print(df_bank.dtypes)
print(f"\nMissing values:")
print(df_bank.isnull().sum())

### Preprocessing and Feature Engineering

In [ ]:
# Identify target variable (typically 'y' for yes/no subscription)
# Assuming the last column is the target
target_col_bank = df_bank.columns[-1]
print(f"Target column: {target_col_bank}")

# Separate features and response
X_bank = df_bank.drop(columns=[target_col_bank])
y_bank = df_bank[target_col_bank]

print(f"\nFeatures shape: {X_bank.shape}")
print(f"Target shape: {y_bank.shape}")
print(f"Target variable distribution:")
print(y_bank.value_counts())

# Get continuous and categorical columns
bank_continuous = X_bank.select_dtypes(include=[np.number]).columns.tolist()
bank_categorical = X_bank.select_dtypes(include=['object']).columns.tolist()

print(f"\nContinuous features: {bank_continuous}")
print(f"Categorical features: {bank_categorical}")

In [ ]:
# Debug: Check raw data
print("DEBUG - Raw X_bank after drop:")
print(f"Shape: {X_bank.shape}")
print(f"Columns: {X_bank.columns.tolist()}")
print(f"Dtypes:\n{X_bank.dtypes}")

# Scale continuous features with MinMaxScaler
if bank_continuous:
    print(f"Scaling continuous features: {bank_continuous}")
    X_bank[bank_continuous] = MinMaxScaler().fit_transform(X_bank[bank_continuous])
else:
    print("No continuous features found")

print(f"\nAfter scaling - Shape: {X_bank.shape}")

# Encode categorical variables to numeric (one-hot encoding)
if bank_categorical:
    print(f"Encoding categorical features: {bank_categorical}")
    X_bank = pd.get_dummies(X_bank, columns=bank_categorical, drop_first=True, dtype=float)
else:
    print("No categorical features found - converting to float")
    X_bank = X_bank.astype(float)

print(f"After encoding - Shape: {X_bank.shape}")
print(f"Columns after encoding: {X_bank.columns.tolist()}")

# Ensure all values are numeric and non-negative (chi2 requirement)
X_bank = X_bank.astype(float)
X_bank = X_bank.fillna(0)
X_bank = X_bank.clip(lower=0)

# Encode target variable if categorical
if y_bank.dtype == 'object':
    y_bank = (y_bank == y_bank.unique()[1]).astype(int)

print(f"\nFinal Features after encoding shape: {X_bank.shape}")
print(f"Features dtype: {X_bank.dtypes.unique()}")
print(f"Features min: {X_bank.min().min()}, max: {X_bank.max().max()}")
print(f"Target variable distribution:")
print(y_bank.value_counts())

### Feature Selection

In [ ]:
# Final verification before chi2
print("Before chi2 - Data Info:")
print(f"X_bank shape: {X_bank.shape}")
print(f"X_bank dtypes unique: {X_bank.dtypes.unique()}")
print(f"X_bank has NaN: {X_bank.isnull().any().any()}")
print(f"X_bank has inf: {np.isinf(X_bank.values).any()}")
print(f"X_bank min: {X_bank.min().min()}, max: {X_bank.max().max()}")
print(f"y_bank unique values: {np.unique(y_bank)}")

# Convert to numpy array to avoid any pandas issues
X_bank_array = np.asarray(X_bank, dtype=np.float64)

# Feature selection with SelectPercentile
X_bank_selected = SelectPercentile(score_func=chi2, percentile=10).fit_transform(X_bank_array, y_bank)
print(f"\nFeatures after selection shape: {X_bank_selected.shape}")

# Split train and validation data
X_bank_train, X_bank_valid, y_bank_train, y_bank_valid = train_test_split(
    X_bank_selected, y_bank, train_size=0.7, stratify=y_bank
)

print(f"Training set size: {X_bank_train.shape[0]}")
print(f"Validation set size: {X_bank_valid.shape[0]}")

### Scaling and Model Training

In [ ]:
# Scale to mean = 0 and standard deviation = 1
bank_scaler = preprocessing.StandardScaler().fit(X_bank_train)
X_bank_train_scaled = bank_scaler.transform(X_bank_train)
X_bank_valid_scaled = bank_scaler.transform(X_bank_valid)

# Create and train KNN model
knn_bank = KNeighborsClassifier(n_neighbors=7)
knn_bank.fit(X_bank_train_scaled, y_bank_train)

# Cross-validation
cv_scores_bank = cross_val_score(knn_bank, X_bank_train_scaled, y_bank_train, cv=5)
print(f"Cross-validation scores: {cv_scores_bank}")
print(f"Mean CV score: {cv_scores_bank.mean():.4f} (+/- {cv_scores_bank.std():.4f})")

# Calculate accuracy
acc_bank_train = knn_bank.score(X_bank_train_scaled, y_bank_train) * 100
acc_bank_valid = knn_bank.score(X_bank_valid_scaled, y_bank_valid) * 100
print(f"\nAccuracy on training data: {acc_bank_train:.2f}%")
print(f"Accuracy on validation data: {acc_bank_valid:.2f}%")

### Model Evaluation

In [ ]:
# Predictions
y_bank_pred = knn_bank.predict(X_bank_valid_scaled)

# Confusion Matrix
print("Confusion Matrix:")
cm_bank = pd.crosstab(y_bank_valid, y_bank_pred, rownames=['Actual'], colnames=['Predicted'])
print(cm_bank)

# Visualization
cm_bank_values = confusion_matrix(y_bank_valid, y_bank_pred, labels=knn_bank.classes_)
plt.figure(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm_bank_values, display_labels=['No Subscription', 'Subscription']).plot(cmap='BuPu')
plt.title('Bank Marketing - Confusion Matrix')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

---
## Summary and Comparison

In [ ]:
# Create summary comparison table
summary_data = {
    'Dataset': ['Telco Churn', 'Online Shoppers', 'Bank Marketing'],
    'Original Features': [X_telco.shape[1], X_online.shape[1], X_bank.shape[1]],
    'Selected Features': [X_telco_selected.shape[1], X_online_selected.shape[1], X_bank_selected.shape[1]],
    'Train Accuracy': [f"{acc_telco_train:.2f}%", f"{acc_online_train:.2f}%", f"{acc_bank_train:.2f}%"],
    'Valid Accuracy': [f"{acc_telco_valid:.2f}%", f"{acc_online_valid:.2f}%", f"{acc_bank_valid:.2f}%"],
    'Mean CV Score': [
        f"{cv_scores_telco.mean():.4f}",
        f"{cv_scores_online.mean():.4f}",
        f"{cv_scores_bank.mean():.4f}"
    ]
}

summary_df = pd.DataFrame(summary_data)
print("\nModel Performance Summary:")
print(summary_df.to_string(index=False))